# Chapter 3 — where should the context budget go?

Prior work asks **how much** context helps. We fix the amount and ask **where it
should go**, then price every answer in tokens.

> At a **fixed** context budget, does it matter **how specifically** the context
> is matched to the element being completed?

**Task: link prediction**, 50-way filtered, both directions, on an **inductive**
split — test entities are unseen, so context is the *only* thing the model has.

---

### ⚠️ Read this before running anything

Six things were wrong in the first version of this pipeline. All are fixed, and
the fixes change what you are allowed to conclude:

| | was | now |
|---|---|---|
| **F1** | `fp` and `fn` incremented on the same class, so P = R = F1 = Hits@1 | real confusion matrix via `--task relation` |
| **verdicts** | declared at ±0.005 MRR, well under the ±0.02 noise floor | paired bootstrap; CI must exclude 0 |
| **candidates** | sampled inline, identical across policies only by luck | frozen to disk, identical by construction |
| **filtering** | train ∪ test | train ∪ **valid** ∪ test |
| **direction** | tail only | tail **and** head |
| **untuned** | never run | reported as its own row |

And two bugs found while testing those fixes:

- ★★ **Inductive queries had no neighbours at all.** Neighbours were read from
  `train` only — but inductive test entities are unseen *by definition*, so the
  block that S1/S2/S4/S5 all discriminate on was never emitted. Most of the
  ladder produced byte-identical prompts to the baseline. **This would have been
  reported as "specificity does not pay" when nothing was ever allocated.**
- **The 468× bug from Chapter 1, reintroduced** — the graph index was rebuilt
  from scratch on every call.

Both are pinned by regression tests in `chapter3/test_pipeline.py`.

## 0 · Setup and the run tracker


In [ ]:
REPO_URL = 'https://github.com/lynda-lagh/contribution-.git'
DEST     = '/kaggle/working/repo'
DATASET  = 'WN18RR-ind'      # pipeline validation. FB15k-237-ind carries the claim
BUDGETS  = [0, 30, 60, 120, 240]
LIMIT    = 300               # ranking queries. 300 x 50 = 15k passes ~ 12 min

import os, sys, json, glob, time, socket, subprocess
from pathlib import Path

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('No network. Settings -> Internet -> ON.')

def sh(*cmd, check=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise SystemExit(f"$ {' '.join(cmd)}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

before = sh('git','-C',DEST,'rev-parse','--short','HEAD', check=False) or '(none)'
if os.path.isdir(f'{DEST}/.git'):
    sh('git','-C',DEST,'fetch','--depth','1','origin','main')
    sh('git','-C',DEST,'reset','--hard','FETCH_HEAD')
else:
    sh('git','clone','--depth','1',REPO_URL,DEST)
os.chdir(DEST); sys.path.insert(0, DEST)
print('repo', before, '->', sh('git','-C',DEST,'rev-parse','--short','HEAD'))
print('    ', sh('git','-C',DEST,'log','-1','--pretty=%s'))

# ★ live feedback helpers — bars, timings, leaderboards, state panel
from chapter3.live import (step, run, bar, delta, leaderboard,
                           improvement_panel, panel, CHANCE, load_cells)
print(f'\nlive helpers loaded   ·   50-way chance MRR = {CHANCE:.4f}')

# ── Hugging Face token (optional: raises rate limits, no gated models needed) ─
#  ⚠️ NEVER paste a token into a cell — the notebook is committed to a public
#     repo and the token would go with it. Kaggle Secrets keeps it out of the file.
#     Add-ons -> Secrets -> label HF_TOKEN -> paste -> attach to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = _tok
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _tok
    print('hf: ✅ authenticated via Kaggle Secrets')
except Exception as _e:
    print(f'hf: ⬜ anonymous ({type(_e).__name__}) — fine for Qwen2.5-1.5B-Instruct,')
    print('    which is public. Only download rate limits are affected.')


import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (phase 0 only)'
print('gpu:', gpu)

In [ ]:
# ★ FINAL SUMMARY — everything, in one screen
panel(DATASET)
leaderboard('results', DATASET)
improvement_panel('results', DATASET)

# efficiency: MRR per 1,000 context tokens — what makes this Chapter 3
import json, glob, re
print(f"\n{'='*74}\n  EFFICIENCY — MRR per 1,000 context tokens\n{'='*74}")
eff = {}
for f in glob.glob(f'results/ch3_{DATASET}_*_B*_tail_tuned.json'):
    m = re.search(rf'ch3_{DATASET}_(.+)_B(\d+)_tail_tuned\.json$', f)
    if not m: continue
    d = json.loads(open(f).read())
    v = d['cost'].get('MRR_per_1k_tokens')
    if v: eff[(m.group(1), int(m.group(2)))] = v
if eff:
    hi = max(eff.values())
    for (p, b), v in sorted(eff.items(), key=lambda kv: -kv[1])[:12]:
        print(f'  {p:15s} B={b:<4d} {v:>8.4f}  {bar(v, 0, hi, 24)}')
    print('\n  ★ A policy only means something relative to what it spent.')

## Phase 0 — free. No GPU quota.

Everything here runs on CPU. Do it in a **CPU-only session** so it costs nothing
against your GPU hours.

★ Start with the smoke test below: it exercises the entire pipeline on a
synthetic graph in about two seconds and will tell you immediately if anything
is broken, before you spend a single GPU-minute.

In [ ]:
!pip install -q nltk pyyaml

# ★ PREFLIGHT FIRST. Checks that the pulled repo is internally consistent.
#   A partial commit (chapter3/ pushed, src/ not) makes new tests run against
#   old library code; the symptom is a bare AttributeError that names a
#   symptom, not a cause. This names the cause and the exact git command.
import subprocess
if subprocess.call('python -m chapter3.preflight', shell=True) != 0:
    raise SystemExit('✋ repository is inconsistent — commit the files listed above')

# ★ THE SMOKE TESTS — whole pipeline on a synthetic graph, no GPU, ~10 seconds.
checks = [
    ('pipeline end-to-end (49 checks)', 'python -m chapter3.test_pipeline'),
    ('allocator unit tests (27, x3 seeds)', 'python -m chapter3.test_chapter3 --repeat 3'),
    ('bootstrap estimator self-check', 'python -m chapter3.stats --demo'),
    ('bar rendering sanity', 'python -m chapter3.live'),
]
results = []
for name, cmd in checks:
    with step(name):
        results.append((name, subprocess.call(cmd, shell=True)))

print('\n' + '='*70)
for name, rc in results:
    print(f"  {'✅ PASS' if rc==0 else '❌ FAIL'}   {name}")
print('='*70)
if any(rc for _, rc in results):
    raise SystemExit('✋ a smoke test failed — do not continue')
print('all green — safe to spend GPU')

### 0b · get the data

The CATS repository is **code only** — 14 files, all `.py`/`.pdf`/`.png`. Their
README points the data at a Google Drive folder, so cloning can never produce a
split.

Run **cell A once**, download the zip, and save it as a Kaggle Dataset. Every
later session then uses **cell B** and skips the 2 GB download.

★ `inductive_graph.txt` becomes `valid.tsv`: it is the observable graph for
unseen entities, so it is both filtered against during ranking and read by
`GraphIndex` as the support from which neighbour blocks are built. Discarding it
would leave every test entity with no context.

★ `ranking_tail.txt` holds exactly 50 candidates per query — CATS's own sets. We
adopt them instead of sampling our own, which makes the numbers directly
comparable to CATS and RealKGC.

In [ ]:
# ── CELL A · locate the CATS data (mounted dataset preferred, Drive as fallback)
import subprocess, sys, shutil
from pathlib import Path

URL = 'https://drive.google.com/drive/folders/17C3BsllCWy_TK3B5WwCjxPQo2heuLJPz'

def find_cats():
    """
    Find the folder holding the CATS dataset directories.

    ⚠️ Located by CONTENT, not by path. Kaggle mounts a dataset at
       /kaggle/input/<slug>/, but the depth inside depends on how the zip was
       built, so hard-coding a path fails silently on the next upload.
       `inductive_graph.txt` is the unambiguous marker.
    """
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    for root in roots:
        if not root.exists():
            continue
        for marker in root.rglob('inductive_graph.txt'):
            # .../<datasets>/<NAME-subset-inductive>/inductive_graph.txt
            return marker.parent.parent
    return None

CATS = find_cats()
if CATS is None:
    print('no mounted data found — downloading from Google Drive')
    DEST = Path('/kaggle/working/CATS-data')
    subprocess.run([sys.executable,'-m','pip','install','-q','-U','gdown'], check=False)
    DEST.mkdir(parents=True, exist_ok=True)
    for args in (['--folder',URL,'-O',str(DEST),'--remaining-ok'],
                 ['--folder',URL,'-O',str(DEST)]):
        subprocess.run([sys.executable,'-m','gdown',*args], text=True)
        if any(DEST.rglob('*')): break
    shutil.make_archive('/kaggle/working/CATS-data','zip',DEST)
    print('\n★ download CATS-data.zip and save it as a Kaggle Dataset,')
    print('  or the next session repeats this 2 GB download.')
    CATS = find_cats()

if CATS is None:
    raise SystemExit('✋ could not locate the CATS data. Add your dataset via '
                     'Add Input, or let the Drive download run.')

print(f'✅ CATS data at:  {CATS}\n')
print(f"{'folder':38s} {'test':>6s} {'rank_tail':>10s}  inductive?")
for p in sorted(CATS.glob('*')):
    if not p.is_dir():
        continue
    nl = lambda f: sum(1 for _ in (p/f).open(encoding='utf-8', errors='replace')) \
                   if (p/f).exists() else 0
    t, rt = nl('test.txt'), nl('ranking_tail.txt')
    ind = '✅ yes' if (p/'inductive_graph.txt').exists() else '—'
    ratio = f'({rt//t}-way)' if t and rt else ''
    print(f'   {p.name:35s} {t:>6,} {rt:>10,}  {ind} {ratio}')

In [ ]:
# ── CELL B · convert to our format, then validate ───────────────────────────
#  WN18RR-ind  : 188 test queries. WordNet ids, so S5 is testable. Weak power.
#  NELL-995-ind: 476 test queries. 2.5x the power, no WordNet hierarchy.
SRC = f'{CATS}/WN18RR-subset-inductive'

run(f'python -m scripts.convert_cats --src {SRC} --out data/{DATASET}',
    'convert CATS -> our format', check=True)

run(f'python -m chapter3.validate --dataset {DATASET}',
    'validate the inductive premise', check=True)

from src.data.loaders import load_kg
kg = load_kg(DATASET, 'data')
tr = {e for t in kg.train for e in (t.head, t.tail)}
te = {e for t in kg.test for e in (t.head, t.tail)}
print(f"\n  entities   {len(kg.ent2txt):,}      relations {len(kg.rel2txt)}")
print(f"  train      {len(kg.train):,}   valid(support) {len(kg.valid):,}   "
      f"test {len(kg.test):,}")
print(f"  ★ test entities unseen in train: {len(te-tr):,}/{len(te):,} "
      f"({len(te-tr)/max(1,len(te)):.0%})")

from chapter3.sources import GraphIndex
idx = GraphIndex(kg)
n = sum(1 for t in kg.test
        if idx.neighbours_of(t.head, (t.head, t.relation, t.tail), 5))
print(f"  ★ support facts available: {idx.n_support:,}")
print(f"  ★ queries WITH a neighbour block: {n}/{len(kg.test)} "
      f"({n/max(1,len(kg.test)):.0%})")
if n == 0:
    raise SystemExit('✋ no query has context — the ladder cannot differ from S0')

panel(DATASET)

### 0c · the S5 decision — three checks that can kill the feature

`S5_semantic` allocates by how *specific* a label's meaning is. It has three cheap
ways of being an illusion:

| | check | if it fails |
|---|---|---|
| 1 | does depth **vary**? | the 95.7% quality-band mistake, again |
| 2 | is depth ≈ **log-degree**? | it is long-tail routing — **P30/KICGPTv2** owns that |
| 3 | is depth ≈ **cluster depth**? | **P31/GS-Quant** derives it free by clustering |

Runs on WN11 too, which is already on disk — no download needed.


In [ ]:
# ★ the three checks that decide whether S5_semantic may be REPORTED at all
run(f'python -m chapter3.profile_specificity --dataset {DATASET} --root data',
    'S5 pre-checks (depth variance · vs degree · vs cluster depth)', est='5 min')

import json
from pathlib import Path
sp = Path('results', f'ch3_specificity_{DATASET}.json')
if sp.exists():
    d = json.loads(sp.read_text()); n = d['checks_passed']
    print(f"\n  {'█'*n}{'·'*(3-n)}   {n}/3 checks passed")
    print('  ✅ S5 will be INCLUDED in the ladder' if n >= 2 else
          '  ⚠️ S5 will be EXCLUDED — reporting the profile as the reason IS a finding')
else:
    print('  ⬜ profiler produced no file — S5 stays excluded')

### 0d · relation descriptions, generated **and gated**

The old pipeline emitted `"the relation 'X' links a subject to an object"` — the
same sentence for every relation. Evidence it was empty: L0→L1 moved train loss by
**0.00025**.

★ The gate is **N9** — the skeleton calls the generated-content quality gate *"the
thesis's central missing piece"*. With 11 relations it can be applied to every one
rather than to a sample. **This is a result that needs no GPU.**

⚠️ generation needs a GPU. Skip in Phase 0 and fold it into the start of Phase 1.


In [ ]:
# GPU step — run at the start of Phase 1, takes ~1 minute for 11 relations
!python -m chapter3.sources --dataset {DATASET} --generate
# CPU-only: re-score descriptions that already exist
# !python -m chapter3.sources --dataset {DATASET} --gate


### 0e · ★ freeze the candidate sets, then build the prompts

**Order matters.** Candidates are frozen *first*, to disk. Every policy then
ranks against those exact negatives — a property of the artefact, not of a lucky
execution order.

The previous version sampled negatives inline from a seeded RNG whose state
advanced per query. That is reproducible only while the seed, the query order
**and the query count** all match. Change `--limit` between two cells and the
two policies silently rank against different negatives, and the whole
"matched cost" claim is false with nothing visibly wrong.

`chapter3.data` then prints two guards you should read:

- `[data] N/200 sampled queries have a neighbours block` — ★ if this is **0**,
  stop. It means unseen entities have no context to allocate and the ladder
  cannot differ from the baseline.
- `[guard] B=120 tail: 7/7 policies produce DISTINCT prompts ✓` — if two policies
  are byte-identical, that cell measures nothing.

In [ ]:
# ── candidates: CATS already shipped them, so only build what is missing ────
import glob
have = sorted(glob.glob(f'data/{DATASET}/candidates_*way_s*.json'))
if have:
    print('✅ using CATS\'s own candidate sets — directly comparable to their table')
    for f in have:
        import json as _j
        n = len(_j.load(open(f)))
        print(f'   {Path(f).name:38s} {n:>5,} queries')
else:
    print('no shipped candidates — sampling our own (still valid, less comparable)')
    run(f'python -m chapter3.candidates --dataset {DATASET} --direction both '
        f'--n-way 50 --limit 2000', 'freeze candidate sets', check=True)

# build prompts for every policy x budget x direction
run(f'python -m chapter3.data --dataset {DATASET} --all --budget 30 60 120 240 '
    f'--direction both --limit {LIMIT}', 'build prompts', est='4 min', check=True)

panel(DATASET)

In [ ]:
# ★ WHERE DID EACH POLICY SPEND ITS BUDGET?  (a paper table)
import json, glob
from pathlib import Path
from collections import Counter

B = 120
rows = {}
for f in sorted(glob.glob(f'data/{DATASET}/built/*_B{B}_tail/allocations.json')):
    pid = Path(f).parent.name.replace(f'_B{B}_tail','')
    if pid == 'ORACLE':          # retired: kept nothing, spent 0 — skip
        continue
    A = json.loads(Path(f).read_text())
    tot = Counter()
    for a in A:
        for k, v in a['tokens_by_kind'].items():
            tot[k] += v
    n = max(1, len(A))
    rows[pid] = ({k: v/n for k, v in tot.items()},
                 sum(a['utilisation'] for a in A)/n,
                 sum(1 for a in A if a.get('dropped'))/n)

kinds = sorted({k for r, _, _ in rows.values() for k in r})
print(f'  MEAN TOKENS PER QUERY AT B={B}\n')
print(f"  {'policy':15s} " + "".join(f'{k[:11]:>12s}' for k in kinds) + f"{'util':>7s}{'drop':>7s}")
for pid, (spend, util, drop) in sorted(rows.items()):
    line = f'  {pid:15s} ' + "".join(f'{spend.get(k, 0):>12.1f}' for k in kinds)
    print(line + f'{util:>7.0%}{drop:>7.0%}')

print('\n  spend profile (each policy normalised to its own budget):')
for pid, (spend, _, _) in sorted(rows.items()):
    tot = sum(spend.values()) or 1
    seg = ''.join(('█' if i % 2 == 0 else '▓') *
                  max(0, int(round(spend.get(k, 0) / tot * 40)))
                  for i, k in enumerate(kinds))
    print(f'  {pid:15s} {seg}')
print('  legend: ' + ' | '.join(f'{"█" if i%2==0 else "▓"} {k}'
                                for i, k in enumerate(kinds)))

# ── what the table says ─────────────────────────────────────────────────────
if rows:
    _, u, d = next(iter(rows.values()))
    print(f'\n  utilisation {u:.0%} · blocks dropped on {d:.0%} of queries')
    if d > 0.8 and u > 0.95:
        print('  ✅ the budget BINDS on nearly every query — the allocation decision')
        print('     is real, and any MRR difference is attributable to it.')
    elif d < 0.2:
        print('  ⚠️ the budget rarely binds: nearly everything fits, so priority')
        print('     order barely matters. Report the TIGHT budgets as informative.')

    # which pairs are indistinguishable?
    sig = {pid: tuple(round(s.get(k, 0), 1) for k in kinds)
           for pid, (s, _, _) in rows.items()}
    same = {}
    for pid, v in sig.items():
        same.setdefault(v, []).append(pid)
    dup = [v for v in same.values() if len(v) > 1]
    if dup:
        print('\n  ≡ identical spend profiles:')
        for g in dup:
            print(f'      {", ".join(g)}')
        print('    These policies route on features that do not vary on this graph.')
        print('    Pre-registered in policies.INTERPRETATION — report, do not hide.')

## Phase 1 — the go/no-go · ~1.4 GPU-h

One shared model serves every cell. It is trained on a **mixture**: random
policy, random budget and random direction per example, so no policy is out of
distribution at evaluation time. (P28's context-corruption idea, repurposed.)

Then a cheap **go/no-go**: evaluate `S0_uniform` and one contrasting policy at a
single budget and check they differ at all before paying for the full grid.

⚠️ **ORACLE has been retired.** It allocated on `meta['helps']`, which says
whether a block improves *this* query — knowable only by scoring the model with
and without each block, i.e. 2^|blocks| forward passes. `candidate_blocks` never
set the key, so the policy kept nothing, spent 0 tokens at every budget, and
would have reported "no headroom" everywhere for a reason unrelated to the data.

★ The ceiling is now computed **after the fact** by
`report.policy_selection_oracle`: for each query, the best rank achieved by any
policy at that budget. It bounds what a perfect *router over these policies*
could reach, costs no extra GPU, and is a tighter and more honest bound. The
paper must say which quantity it reports.

In [ ]:
# the shared model: one training run serves every (policy, budget) cell
run(f'python -m chapter3.data --dataset {DATASET} --train-mixed --limit {LIMIT}',
    'build mixed training set (random policy + budget + direction)', est='3 min')

rc = run(f'python -m src.train.sft_cli '
         f'--data data/{DATASET}/built/mixed '
         f'--out checkpoints/ch3-{DATASET}-shared '
         f'--run-name ch3-{DATASET}-shared',
         'train the SHARED model', est='35 min')

import json
from pathlib import Path
s = Path('checkpoints', f'ch3-{DATASET}-shared', 'train_summary.json')
if s.exists():
    d = json.loads(s.read_text()); c = d.get('curve', {})
    print(f"\n  train_loss {d['train_loss']:.5f}   {d['train_runtime_s']/60:.1f} min"
          f"   peak VRAM {d['peak_vram_gb']:.1f} GB")
    print(f"  fit verdict: {c.get('verdict','(none)')}")
    ec = c.get('eval_curve', [])
    if ec:
        lo = min(v for _, v in ec); hi = max(v for _, v in ec)
        print('\n  eval loss curve:')
        for stp, v in ec:
            print(f'    step {stp:>5d}  {v:.5f}  ' + bar(hi - v, 0, max(1e-9, hi-lo), 30))
elif rc != 0:
    raise SystemExit('✋ training failed — read the traceback above')

In [ ]:
import subprocess, time
t0 = time.time()
cmd = (f'python -m src.train.sft_cli --data data/{DATASET}/built/mixed '
       f'--out checkpoints/ch3-{DATASET}-shared --run-name ch3-{DATASET}-shared')
print('$', cmd)
rc = subprocess.call(cmd, shell=True)
print(f'rc={rc}  {(time.time()-t0)/60:.1f} min')
# ⚠️ if src.train.sft_cli does not exist, train via the trainer directly:
#    from src.train.sft import train_sft; from src.utils.config import load_config
#    train_sft(load_config('configs/base.yaml'),
#              f'data/{DATASET}/built/mixed', f'checkpoints/ch3-{DATASET}-shared')

In [ ]:
# ★ ONE evaluation helper. Every call prints its own result AND the leaderboard,
#   so no cell ever finishes without telling you where you now stand.
import subprocess, json, time
from pathlib import Path
ADAPTER = f'checkpoints/ch3-{DATASET}-shared'

def rpath(pol, b, direction='tail', tag='tuned', task='link'):
    sfx = '' if task == 'link' else '_rel'
    return Path('results', f'ch3_{DATASET}_{pol}_B{b}_{direction}_{tag}{sfx}.json')

def done(pol, b, direction='tail', tag='tuned', task='link'):
    return rpath(pol, b, direction, tag, task).exists()

def ev(pol, b, adapter=ADAPTER, direction='tail', task='link', show=True):
    tag = 'tuned' if adapter else 'untuned'
    if done(pol, b, direction, tag, task):
        print(f'  ⏭  skip {pol} B={b} {direction} {tag} {task} (already done)')
        return 0
    cmd = (f'python -m chapter3.evaluate --dataset {DATASET} --policy {pol} '
           f'--budget {b} --limit {LIMIT} --direction {direction} --task {task}'
           + (f' --adapter {adapter}' if adapter else ''))
    rc = run(cmd, f'{pol}  B={b}  {direction}  {tag}  {task}', est='12 min')
    p = rpath(pol, b, direction, tag, task)
    if rc == 0 and p.exists() and show:
        d = json.loads(p.read_text()); m = d['ranking']['MRR']
        base = rpath('S0_uniform', b, direction, tag, task)
        print(f'\n  ➜ MRR {m:.4f}  {bar(m, CHANCE, max(m*1.1, CHANCE+1e-6))}'
              f'   vs chance {delta(m, CHANCE)}')
        if base.exists() and pol != 'S0_uniform':
            b0 = json.loads(base.read_text())['ranking']['MRR']
            print(f'    vs S0 baseline {b0:.4f}   {delta(m, b0)}')
        if m <= CHANCE + 0.005:
            print('    ⚠️ AT CHANCE — this policy learned nothing at this budget')
    return rc

# ── the go / no-go ──────────────────────────────────────────────────────────
#  ⚠️ ORACLE is retired. It kept no blocks (meta['helps'] is never set) so it
#     spent 0 tokens and silently reproduced the B=0 floor — it would have said
#     "no headroom" at every budget for a reason unrelated to the data.
#     The ceiling is now computed POST HOC from the policy results themselves:
#     for each query, the best rank any policy achieved. Costs no GPU.
#
#  So the go/no-go is: run S0 and one contrasting policy at one budget, and
#  check they differ at all before paying for the full grid.
for pol in ['S0_uniform', 'S4_instance']:
    ev(pol, 120)

leaderboard('results', DATASET, budget=120)

import json
a, b = rpath('S0_uniform', 120), rpath('S4_instance', 120)
if a.exists() and b.exists():
    from chapter3.stats import align, paired_bootstrap, fmt_diff, verdict
    ra = json.loads(a.read_text()); rb = json.loads(b.read_text())
    try:
        x, y, n = align(rb.get('rows', []), ra.get('rows', []))
        t = paired_bootstrap(x, y)
        print(f"\n  S4 − S0 at B=120:  {fmt_diff(t)}   (n={n} paired)")
        print(f"  {verdict(t, 'S4_instance', 'S0_uniform')}")
        print(f"\n  ➤ {'proceed to the full grid' if abs(t['diff']) > 0.01 or t['significant'] else 'the two policies are close; the grid may return a null result — still worth running, but expect to report it as one'}")
    except ValueError as e:
        print('  cannot pair:', e)

## Phase 2 — the claim · ~4 GPU-h

Three rows carry the whole chapter: **S0** (uniform), **R** (random control) and
the best policy. Everything else is detail.

★ **R is what makes the result interpretable.** Same budget, same action mix,
decisions shuffled. If the best policy ≈ R, the *decisions* added nothing and
only the budget mattered — a clean negative result, and the specificity analogue
of the field's "more context is not better". Without R, "S4 ≈ S0" is ambiguous
between *specificity does not pay* and *our policy is bad*, and a reviewer will
say exactly that.

In [ ]:
import time
CORE = ['S0_uniform', 'R_random', 'S4_instance']
t_start = time.time()
todo = [(p, B) for B in [30, 60, 120, 240] for p in CORE if not done(p, B)]
print(f'  {len(todo)} cells to run   ·   ~{len(todo)*12} min estimated\n')

for n, (pol, B) in enumerate(todo, 1):
    print(f'\n{"#"*74}\n#  {n}/{len(todo)}   {pol}  B={B}   '
          f'elapsed {(time.time()-t_start)/60:.0f} min\n{"#"*74}')
    ev(pol, B)

# the B=0 floor — no context, identical for every policy. Evaluate once.
ev('S0_uniform', 0)

leaderboard('results', DATASET)
improvement_panel('results', DATASET)
panel(DATASET)

### 2b · ★ the three rows that answer the obvious reviewer questions

Each is cheap, and each closes a hole a referee will otherwise open.

**Untuned.** Same evaluation, no adapter. If allocation pays *without*
fine-tuning, the claim is about the **context**, not about our training recipe —
much stronger, and much cheaper for someone else to replicate.

**Head direction.** CATS and RealKGC both report `(h, r, ?)` and `(?, r, t)`.
A one-directional table invites the assumption that the easy side was chosen.

**Relation prediction.** ★ The only place a genuine F1 and a confusion matrix
exist. On the link task, per-relation precision, recall and F1 are all
*algebraically identical* to Hits@1 — a miss is a miss, there is no wrong class
to confuse. Predicting `(h, ?, t)` over the relation vocabulary gives real
classes, so `_hypernym` mistaken for `_hyponym` becomes a visible, meaningful
error: **a directional mistake that more tokens will not fix but the right
tokens might.**

In [ ]:
# ★ 1 · UNTUNED — is the gain the context, or our training recipe?
for B in [60, 120]:
    for pol in ['S0_uniform', 'R_random', 'S4_instance']:
        ev(pol, B, adapter=None)

print('\n' + '='*74 + '\n  TUNED vs UNTUNED\n' + '='*74)
import json
for B in [60, 120]:
    for pol in ['S0_uniform', 'S4_instance']:
        t, u = rpath(pol, B, tag='tuned'), rpath(pol, B, tag='untuned')
        if t.exists() and u.exists():
            tm = json.loads(t.read_text())['ranking']['MRR']
            um = json.loads(u.read_text())['ranking']['MRR']
            print(f'  {pol:14s} B={B:<4d} untuned {um:.4f}  tuned {tm:.4f}  '
                  f'{delta(tm, um)}')

# ★ 2 · HEAD direction — CATS and RealKGC report both
for pol in ['S0_uniform', 'R_random', 'S4_instance']:
    ev(pol, 120, direction='head')

# ★ 3 · RELATION prediction — the only real confusion matrix
for pol in ['S0_uniform', 'S4_instance']:
    ev(pol, 120, task='relation')

print('\n  head direction:'); leaderboard('results', DATASET, direction='head')
panel(DATASET)

## Phase 3 — the ladder (optional) · ~1.7 GPU-h

S1–S3 at one budget, and **S5 only if `profile_specificity` passed all three checks**.


In [ ]:
import json
from pathlib import Path
LADDER = ['S1_property', 'S2_type', 'S3_quality']

# ★ S5 is GATED — only reported if its profiler passed
sp = Path('results', f'ch3_specificity_{DATASET}.json')
if sp.exists():
    d = json.loads(sp.read_text()); n = d['checks_passed']
    print(f"  S5 gate: {'█'*n}{'·'*(3-n)}  {n}/3")
    if n >= 2:
        LADDER.append('S5_semantic'); print('  ✅ S5 INCLUDED')
    else:
        print('  ⚠️ S5 EXCLUDED — report the profile as the reason; that IS a finding')
else:
    print('  ⚠️ S5 EXCLUDED — profiler not run (cell 0c)')

for pol in LADDER:
    ev(pol, 120)

leaderboard('results', DATASET, budget=120)
improvement_panel('results', DATASET)

## 4 · Results — every table

`report.py` prints seven views. **Read view 2 first**: it is the whole claim.

★★ **Every verdict is now backed by a paired bootstrap.** The previous version
declared a result at ±0.005 MRR, but the standard error at n=300 is around
±0.02 — so every one of those verdicts could have flipped on the seed. A
difference is now a result only when its 95% interval **excludes zero**, and
otherwise it is reported as *unmeasurable*, which is a different sentence from
"a small gain" and the only honest one.

Pairing is what makes this affordable: every policy is scored on the **same
queries** against the **same frozen candidates**, so we resample the per-query
*difference* and query difficulty cancels out. That roughly halves the interval
versus an unpaired test.

In [ ]:
# every view: grid with CIs, the anchors with PAIRED bootstrap, efficiency,
# calibration, the confusion matrix, per-relation, tuned-vs-untuned, both directions
run(f'python -m chapter3.report --dataset {DATASET} --compare-untuned --both-directions',
    'full report (7 views, paired bootstrap)', est='1 min')

In [ ]:
# ★ the head direction on its own, with its own statistics
!python -m chapter3.report --dataset {DATASET} --direction head

### ★ the qualitative table — the half-page that persuades

The grid *proves* the effect; this *shows* it. Same query, same budget, two
policies, and the rank each produced.

⚠️ Cases are chosen by a **stated rule** (largest improvement), never by hand —
a hand-picked example is an anecdote, a rule-selected one is evidence. Run it
with `--worst` too: a case study containing only wins reads as advocacy, and
reviewers of CATS-adjacent papers look for exactly that.

In [ ]:
import subprocess
def qual(*extra):
    cmd = ['python', '-m', 'chapter3.qualitative', '--dataset', DATASET,
           '--budget', '120', '--a', 'S0_uniform', '--b', 'S4_instance',
           '--only-disagreements', *extra]
    print('$', ' '.join(cmd), flush=True)
    return subprocess.call(cmd)

qual('--n', '3', '--latex')      # the wins, with LaTeX ready to paste
qual('--n', '2', '--worst')      # ★ and the regressions — report these too

In [ ]:
# ★ TWO FIGURES: the improvement bars, and the cost curve.
import json, glob, re
import matplotlib.pyplot as plt
import numpy as np

cells = {}
for f in glob.glob(f'results/ch3_{DATASET}_*_B*_tail_tuned.json'):
    m = re.search(rf'ch3_{DATASET}_(.+)_B(\d+)_tail_tuned\.json$', f)
    if not m: continue
    d = json.loads(open(f).read())
    cells[(m.group(1), int(m.group(2)))] = d

if not cells:
    print('no results yet')
else:
    buds = sorted({b for _, b in cells})
    pols = sorted({p for p, _ in cells})
    B = 120 if 120 in buds else buds[-1]

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))

    # ---- (1) improvement bars at one budget, drawn FROM CHANCE ----------
    rows = [(p, cells[(p, B)]) for p in pols if (p, B) in cells]
    rows.sort(key=lambda kv: kv[1]['ranking']['MRR'])
    names = [p for p, _ in rows]
    vals  = [d['ranking']['MRR'] for _, d in rows]
    errs  = [[d['ranking']['MRR'] - (d.get('MRR_ci') or {}).get('lo', d['ranking']['MRR'])
              for _, d in rows],
             [(d.get('MRR_ci') or {}).get('hi', d['ranking']['MRR']) - d['ranking']['MRR']
              for _, d in rows]]
    cols = ['#999' if n == 'R_random'
            else '#1f77b4' if n == 'S0_uniform' else '#2ca02c' for n in names]
    ax[0].barh(names, vals, xerr=errs, color=cols, capsize=3)
    ax[0].axvline(CHANCE, color='r', ls='--', lw=1.4, label=f'chance {CHANCE:.4f}')
    s0 = cells.get(('S0_uniform', B))
    if s0:
        ax[0].axvline(s0['ranking']['MRR'], color='#1f77b4', ls=':', lw=1.4,
                      label='S0 uniform')
    ax[0].set_xlabel('MRR (50-way, filtered)')
    ax[0].set_title(f'{DATASET} — allocation at B={B} tokens\n'
                    f'error bars: 95% bootstrap CI')
    ax[0].legend(fontsize=8); ax[0].grid(axis='x', alpha=.3)

    # ---- (2) the cost curve --------------------------------------------
    for p in pols:
        pts = sorted((cells[(p, b)]['cost']['mean_context_tokens'],
                      cells[(p, b)]['ranking']['MRR']) for b in buds if (p, b) in cells)
        if not pts: continue
        st = dict(marker='o', ms=4)
        if   p == 'R_random':   st.update(ls=':',  color='#999', label='R random (control)')
        elif p == 'S0_uniform': st.update(lw=2.5,  color='#1f77b4', label='S0 uniform')
        else:                   st.update(label=p)
        ax[1].plot([x for x,_ in pts], [y for _,y in pts], **st)
    ax[1].axhline(CHANCE, color='r', ls='--', lw=1.2, label='chance')
    ax[1].set_xlabel('mean context tokens per query')
    ax[1].set_ylabel('MRR (50-way, filtered)')
    ax[1].set_title('ranking quality against context cost')
    ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)

    plt.tight_layout()
    plt.savefig(f'results/ch3_figure_{DATASET}.png', dpi=160)
    plt.show()
    print(f'saved results/ch3_figure_{DATASET}.png')
    print('★ Overlapping error bars do NOT prove equivalence — the PAIRED tests in')
    print('  report.py view 2 are strictly more sensitive. Read those for verdicts.')

In [ ]:
tracker()


## 5 · Backup

⚠️ `/kaggle/working` does not survive an interactive session ending. Download
before closing the tab, or commit the notebook.


In [ ]:
import datetime, subprocess
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
out = f'/kaggle/working/ch3_{stamp}.zip'

# ★ candidates_*.json is included ON PURPOSE: without it the results are not
#   reproducible, because a reader cannot re-derive the negatives.
subprocess.call(
    f'zip -qr {out} results/ data/*/built/*/allocations.json '
    f'data/*/relation_descriptions*.json data/*/candidates_*.json', shell=True)
print('wrote', out)

subprocess.call('python -m scripts.export_adapters --zip --prune', shell=True)
subprocess.call('du -sh results checkpoints 2>/dev/null', shell=True)